# STAIR-REEC / versionICAART — controlled Kaggle training

Static fixed-support calibration of the V8 optimizer graph; no new learned parameters.
Specification: `docs/giai_doan_6/STAIR_versionICAART_Report.md`.

User halves are disjoint, not guaranteed statistically independent. Positive association is
not a causal preference label. Historical seed-1 results are references, not matched reruns.
Run the setup cells in order. Training and tests require explicit opt-in below.


## 1. Workspace and dependency checks
Use a single repository root and explicit subprocess working directory. Existing source is
never reset or silently updated. Set `INSTALL_DEPENDENCIES=True` only when needed.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, hashlib

WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
DATASETS = ['baby', 'sports', 'electronics']
INSTALL_DEPENDENCIES = False
RUN_TESTS = False
RUN_TRAINING = False
AUTO_DOWNLOAD_DATA = False
DATASET_SOURCES = {}  # Optional: {'sports': '/kaggle/input/.../Amazon2014Sports_550_MMRec'}

roots = [Path(os.environ['STAIR_REPO'])] if os.environ.get('STAIR_REPO') else [
    Path.cwd(), WORK_ROOT / 'STAIR-Enhanced', Path.home() / 'STAIR-Enhanced',
]
REPO_ROOT = next((p.resolve() for p in roots if (p / 'main_stair_icaart.py').is_file()), None)
if REPO_ROOT is None:
    target = WORK_ROOT / 'STAIR-Enhanced'
    if target.exists():
        raise FileNotFoundError(f'{target} exists but lacks main_stair_icaart.py; upload the reviewed source or set STAIR_REPO.')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ThanhChuong12/STAIR-Enhanced.git', str(target)], check=True)
    REPO_ROOT = target.resolve()
required = ['main_stair_icaart.py', 'stair_icaart_config.py', 'models/stair_icaart.py',
            'models/stair_icaart_evidence.py', 'models/stair_icaart_graph.py']
missing = [name for name in required if not (REPO_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Repository source is incomplete: {missing}')

SUB_ENV = os.environ.copy()
SUB_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), SUB_ENV.get('PYTHONPATH', '')]))
SUB_ENV['PYTHONIOENCODING'] = 'utf-8'
SUB_ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'freerec==0.9.7',
                    'torch-geometric', 'scipy', 'pyyaml', 'prettytable', 'kagglehub'], check=True)

verify = """
import models.freerec_compat
import torch, freerec, sys
from models.stair_icaart import STAIR_ICAART_Model
from stair_icaart_config import validate_icaart_config
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__, 'FreeRec:', freerec.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Module imports succeeded. This is not a mathematical or training verification.')
"""
subprocess.run([sys.executable, '-c', verify], cwd=REPO_ROOT, env=SUB_ENV, check=True)
print('Repository:', REPO_ROOT)


## 2. Dataset staging
Stage all five required files in a writable `Processed` directory. FreeRec's `chunks`
cache remains writable; read-only input directories are never used as cache directories.
Set an explicit source override if automatic exact-name discovery is ambiguous.


In [ ]:
DATA_ROOT = WORK_ROOT / 'stair_icaart_data'
DATASET_NAMES = {
    'baby': 'Amazon2014Baby_550_MMRec',
    'sports': 'Amazon2014Sports_550_MMRec',
    'electronics': 'Amazon2014Electronics_550_MMRec',
}
REQUIRED_DATA = ('train.txt', 'valid.txt', 'test.txt', 'textual_modality.pkl', 'visual_modality.pkl')
DOWNLOAD_ROOT = None

def complete_dataset(path):
    return path.is_dir() and all((path / name).is_file() for name in REQUIRED_DATA)

def find_dataset(key):
    global DOWNLOAD_ROOT
    name = DATASET_NAMES[key]
    if key in DATASET_SOURCES:
        source = Path(DATASET_SOURCES[key]).resolve()
        if not complete_dataset(source):
            raise FileNotFoundError(f'Incomplete {key} source: {source}')
        return source
    staged = DATA_ROOT / 'Processed' / name
    if complete_dataset(staged):
        return staged.resolve()
    candidates = [REPO_ROOT / 'data' / name, REPO_ROOT / 'data' / 'Processed' / name]
    if Path('/kaggle/input').is_dir():
        candidates.extend(Path('/kaggle/input').rglob(name))
    matches = sorted({p.resolve() for p in candidates if complete_dataset(p)})
    if not matches and AUTO_DOWNLOAD_DATA:
        import kagglehub
        if DOWNLOAD_ROOT is None:
            DOWNLOAD_ROOT = Path(kagglehub.dataset_download('rainyle/stair-datasets-mmrec'))
        matches = sorted({p.resolve() for p in DOWNLOAD_ROOT.rglob(name) if complete_dataset(p)})
    if len(matches) != 1:
        raise FileNotFoundError(f'{key}: expected one complete dataset, found {matches}. Set DATASET_SOURCES[{key!r}].')
    return matches[0]

def stage_dataset(key):
    source = find_dataset(key)
    destination = DATA_ROOT / 'Processed' / DATASET_NAMES[key]
    if destination.is_symlink():
        raise RuntimeError(f'Use a real writable staging directory, not a directory symlink: {destination}')
    destination.mkdir(parents=True, exist_ok=True)
    (destination / 'chunks').mkdir(exist_ok=True)
    for name in REQUIRED_DATA:
        src, dst = source / name, destination / name
        if dst.exists():
            if dst.resolve() != src.resolve():
                raise FileExistsError(f'Existing staged file has another source: {dst}; choose a fresh DATA_ROOT.')
            continue
        try:
            dst.symlink_to(src.resolve())
        except OSError:
            shutil.copy2(src, dst)
    if not os.access(destination / 'chunks', os.W_OK):
        raise PermissionError(f'FreeRec cache is not writable: {destination}')
    print(key, '->', destination)

for key in DATASETS:
    stage_dataset(key)


## 3. Optional tests — not automatic
The review did not execute these tests. Set `RUN_TESTS=True` to run them yourself.
A failure raises an exception; successful imports alone do not imply tests passed.


In [ ]:
if RUN_TESTS:
    subprocess.run([
        sys.executable, '-m', 'pytest', '-v',
        'tests/test_stair_icaart_evidence.py',
        'tests/test_stair_icaart_graph.py',
        'tests/test_stair_icaart_pipeline.py',
    ], cwd=REPO_ROOT, env=SUB_ENV, check=True)
else:
    print('Tests were not run (RUN_TESTS=False).')


## 4. Training launcher
Each attempt has a unique directory containing its command, manifest and log.
A nonzero process exit stops immediately. Native coach telemetry records allocated and
reserved CUDA peaks per training epoch; this does not measure all device/system memory.
Selected TEST parsing requires the native "Load best model" section.


In [ ]:
import re, time, uuid
from datetime import datetime, timezone

RUN_ROOT = WORK_ROOT / 'stair_icaart_runs'
GRAPH_CACHE = DATA_ROOT / '.stair_icaart_graph_cache'
ICAART_RESULTS = {}

NUMBER = r'([-+]?(?:d+(?:.d*)?|.d+)(?:[eE][-+]?d+)?)'
METRICS = ('Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20')

def extract_test_metrics(log_path, artifact_dir=None):
    selected_epoch = None
    metrics = {}
    for line in Path(log_path).read_text(encoding='utf-8', errors='replace').splitlines():
        loaded = re.search(r'Load best model @Epoch\s*:\s*(\d+)', line, re.I)
        if loaded:
            selected_epoch, metrics = int(loaded.group(1)), {}
        event = re.search(r'\bTEST\s+@Epoch\s*:\s*(\d+)', line, re.I)
        if event and selected_epoch is not None and int(event.group(1)) == selected_epoch:
            for metric in METRICS:
                match = re.search(re.escape(metric) + r'\s+Avg\s*:\s*' + NUMBER, line, re.I)
                if match:
                    metrics[metric] = float(match.group(1))
    if selected_epoch is None or set(metrics) != set(METRICS):
        raise RuntimeError(f'Complete selected TEST metrics not found: {log_path}. Inspect the raw log; no zero-valued placeholders were generated.')
    return selected_epoch, metrics

def run_training_icaart(key, delta=1.0, mode='replicated_excess', tau=2.0,
                        seed=1, preprocess_only=False, **extra_args):
    if key not in DATASET_NAMES:
        raise ValueError(f'Unknown dataset: {key}')
    stage_dataset(key)  # Explicit calls work for any dataset, not just a selected default.
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    label = f'{key}_{mode}_delta{delta}_tau{tau}_seed{seed}_{stamp}_{uuid.uuid4().hex[:6]}'
    artifact = RUN_ROOT / label
    artifact.mkdir(parents=True, exist_ok=False)
    log_path = artifact / 'console.log'
    command = [
        sys.executable, '-u', str(REPO_ROOT / 'main_stair_icaart.py'),
        '--config', str(REPO_ROOT / 'configs' / f'{DATASET_NAMES[key].split("_550")[0]}_STAIR_ICAART.yaml'),
        '--root', str(DATA_ROOT), '--seed', str(seed),
        '--evidence-delta', str(delta), '--evidence-mode', mode,
        '--evidence-tau', str(tau), '--artifact-dir', str(artifact),
        '--graph-cache-dir', str(GRAPH_CACHE),
    ]
    if preprocess_only:
        command.append('--preprocess-only')
    reserved = {'root', 'config', 'seed', 'artifact_dir', 'graph_cache_dir',
                'evidence_mode', 'evidence_delta', 'evidence_tau', 'preprocess_only'}
    if reserved.intersection(extra_args):
        raise ValueError(f'Conflicting overrides: {reserved.intersection(extra_args)}')
    for name, value in extra_args.items():
        command.extend(['--' + name.replace('_', '-'), str(value)])
    (artifact / 'launch.json').write_text(json.dumps({
        'command': command, 'cwd': str(REPO_ROOT), 'log': str(log_path),
    }, indent=2), encoding='utf-8')
    print('Running:', subprocess.list2cmdline(command), flush=True)
    started = time.perf_counter()
    with log_path.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=REPO_ROOT, env=SUB_ENV,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', bufsize=1)
        try:
            for line in process.stdout:
                handle.write(line)
                if any(tag in line for tag in ('@Epoch', 'DIAGNOSTICS', 'Dose:', 'Bypass:', 'score > 0', 'Traceback')):
                    print(line.rstrip(), flush=True)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait(timeout=30)
            raise
    if returncode:
        raise RuntimeError(f'Training failed with exit code {returncode}. Full log: {log_path}')
    result = {'dataset': key, 'mode': mode, 'delta': delta, 'tau': tau, 'seed': seed,
              'artifact_dir': str(artifact), 'log_path': str(log_path),
              'elapsed_minutes': (time.perf_counter() - started) / 60,
              'status': 'preprocess_only' if preprocess_only else 'completed'}
    if not preprocess_only:
        result['selected_epoch'], result['test_metrics'] = extract_test_metrics(log_path)
        (artifact / 'selected_test_metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    ICAART_RESULTS[label] = result
    print('Completed:', label, '|', result['status'], '|', artifact)
    return result


## 5. Explicit execution
Default is no training. Enable `RUN_TRAINING=True` after reviewing diagnostics.
All three primary configurations retain 500 epochs. For preprocessing only, call
`run_training_icaart('sports', preprocess_only=True)` explicitly.
Use `mode='v8_exact'` or `delta=0.0` for the V8 control. Choose delta from VALID, not TEST.


In [ ]:
if RUN_TRAINING and 'baby' in DATASETS:
    res_baby = run_training_icaart('baby', delta=1.0, mode='replicated_excess', seed=1)
else:
    print('baby: training not requested.')


In [ ]:
if RUN_TRAINING and 'sports' in DATASETS:
    res_sports = run_training_icaart('sports', delta=1.0, mode='replicated_excess', seed=1)
else:
    print('sports: training not requested.')


In [ ]:
if RUN_TRAINING and 'electronics' in DATASETS:
    res_electronics = run_training_icaart('electronics', delta=1.0, mode='replicated_excess', seed=1)
else:
    print('electronics: training not requested.')


## 6. Current-run results
Only completed primary runs appear here. Historical results are kept in the research
report and must not be presented as freshly matched controls. Missing metrics remain
missing, not zeros. Rerun this cell after a process completes.


In [ ]:
rows = [value for value in ICAART_RESULTS.values()
        if value['status'] == 'completed' and value['mode'] == 'replicated_excess']
print(json.dumps(rows, indent=2))
